# F1 — Experimental validation candidates (roadmap N1–N3)

**Question.** Of the 6,884 predicted variants, which should go on a bench first, and with which
assay? And does the project's own mechanism map support a four-group design
(destabilising-only / rewiring-only / both / neither), or only the two-group version?

**What it does.** Nothing new is derived. F1 reads the frozen outputs of B3 (conformal folding
calls), C4 (the independent specificity axis and its pre-registered decision) and A0 (tumour
context), and turns them into a small, ranked, assay-tagged candidate panel. No model is
retrained, no feature is recomputed, and no number reported by A0–D1 changes. F1 runs after D1
and is not part of the reproduction check.

**The one thing it will not do.** B3's `mechanism` column labels some variants "rewiring" or
"both", but that label rests on rank-matched zero-shot ESM-2 (`bind_source == 'esm_rank_matched'`),
which the project does not claim as a binding prediction (C2 `binding_decision.json`;
B5 `binding_claims_made = False`). F1 does **not** use that column. Specificity candidates come
only from C4's independently trained rewiring score, and only if C4's own pre-registered rule
returned `validated: true`. If it did not, the rewiring panel and the four-group design are not
built: the notebook says so and keeps the two-group design the data support.

**Inputs:** `A0/SH3_cancer_master.tsv`, `B3/atlas_predictions.parquet`,
`C4/specificity_decision.json` (+ `C4/atlas_rewiring.parquet` if validated).
**Outputs:** `experimental_candidates.csv` (per-variant, stays on Drive), `panel_summary.csv`,
`RESULTS_F1.md`. **Runtime:** seconds, CPU. Explanation: `docs/F1_experimental_candidates.md`.

In [ ]:
# ── setup (identical in every notebook) ─────────────────────────────────────────
ROOT   = '/content/drive/MyDrive/sh3_repro'   # the Drive folder from UPLOAD_GUIDE.md
RESUME = True                                  # reuse finished steps after a disconnect

import os, sys, json, time, importlib
from google.colab import drive; drive.mount('/content/drive')
sys.path.insert(0, f'{ROOT}/code')
import sh3_common as C; importlib.reload(C)
import numpy as np, pandas as pd
R = C.Run(ROOT, 'F1', 'Experimental validation candidates', resume=RESUME)
print(C.env_info())

## 1. Load frozen outputs — nothing here is re-derived
`N_PER_PANEL` is the only setting: how many candidates per class a first bench campaign could take on. It is small on purpose; this is a prioritisation aid, not a discovery list.

In [ ]:
N_PER_PANEL = 10   # candidates per class: a starting bench panel, not an exhaustive list

# the atlas must be the verified one from A0 (same MD5 as the original)
a0 = json.load(open(R.res('A0', 'metrics_A0.json')))['metrics']['master_md5']
got = C.md5(R.res('A0', 'SH3_cancer_master.tsv'))
assert got == a0, f'SH3_cancer_master.tsv changed since A0 ran ({got} vs {a0}). Re-run A0.'
MASTER = pd.read_csv(R.res('A0', 'SH3_cancer_master.tsv'), sep='\t', keep_default_na=False, dtype=str)
A = pd.read_parquet(R.res('B3', 'atlas_predictions.parquet'))
A = A.merge(MASTER[['variant_id', 'n_samples_nonhyper', 'top_primary_sites']].drop_duplicates('variant_id'),
            on='variant_id', how='left')
A['nonhyper'] = pd.to_numeric(A.n_samples_nonhyper, errors='coerce').fillna(0) > 0
A['destab'] = A.fold_lo >= 1.0
A['fold_halfwidth'] = A.pred_ddG_fold - A.fold_lo            # the B3 90% half-width

DEC_C4 = json.load(open(R.res('C4', 'specificity_decision.json')))
HAVE_REWIRING = bool(DEC_C4.get('validated', False))
REWIRE_THRESH = np.nan
if HAVE_REWIRING:
    RW = pd.read_parquet(R.res('C4', 'atlas_rewiring.parquet'))[['variant_id', 'rewiring']]
    A = A.merge(RW, on='variant_id', how='left')
    REWIRE_THRESH = A.rewiring.quantile(0.9)                   # same top-decile convention as C4
    A['high_rewiring'] = A.rewiring >= REWIRE_THRESH
else:
    A['rewiring'] = np.nan; A['high_rewiring'] = False
print(f'{len(A):,} atlas variants | destabilising {int(A.destab.sum())} | C4 validated: {HAVE_REWIRING}')

## 2. Panel S — stability candidates (roadmap N1)

Destabilising calls ranked by **how far the lower 90 % bound clears 1 kcal/mol** (`fold_lo`,
highest first), with non-hypermutated occurrences first: a variant seen outside hypermutated
tumours is the more defensible one to spend bench time on. The negative controls are confidently
tolerated variants (|predicted ΔΔG| < 0.3 kcal/mol, narrowest interval), taken **from the same
domains as Panel S where possible**, so that each domain is tested with a positive and a negative.

In [ ]:
stable_pool = A[A.destab].sort_values(['nonhyper', 'fold_lo'], ascending=[False, False])
PANEL_S = stable_pool.head(N_PER_PANEL)

control_pool = A[~A.destab & (A.pred_ddG_fold.abs() < 0.3)].sort_values('fold_halfwidth')
same_dom = control_pool[control_pool.domain_key.isin(PANEL_S.domain_key)].drop_duplicates('domain_key')
PANEL_S_CTRL = pd.concat([same_dom, control_pool.drop(same_dom.index)]).head(N_PER_PANEL)
CTRL_MATCHED = int(PANEL_S_CTRL.domain_key.isin(PANEL_S.domain_key).sum())

STABILITY_ASSAYS = ['thermal denaturation (CD or DSF)', 'proteolysis-based stability assay',
                    'cellular abundance assay', 'ddPCA (AbundancePCA) if the domain can be cloned into that system']
print(f'Panel S: {len(PANEL_S)} destabilising | {len(PANEL_S_CTRL)} controls ({CTRL_MATCHED} from the same domains)')
print(PANEL_S[['variant_id', 'gene', 'pred_ddG_fold', 'fold_lo', 'nonhyper']].to_string(index=False))

## 3. Panel R — specificity (rewiring) candidates (roadmap N2), gated on C4's own rule

Built only if `specificity_decision.json` says `validated: true`. Otherwise this section records
why no panel is produced, instead of falling back to the disclaimed ESM-based label.

In [ ]:
SPECIFICITY_ASSAYS = ['phage display against the peptide library', 'peptide array',
                      'fluorescence polarisation (WT vs mutant against candidate peptides)',
                      'SPR for kinetic confirmation of the top hits',
                      'yeast two-hybrid or a cell-based binding assay']
if HAVE_REWIRING:
    PANEL_R = (A[A.high_rewiring & ~A.destab]
               .sort_values(['nonhyper', 'rewiring'], ascending=[False, False]).head(N_PER_PANEL))
    print(f'Panel R (rewiring, not destabilising): {len(PANEL_R)}')
else:
    PANEL_R = A.iloc[0:0]
    print(f"No specificity panel: C4's pre-registered rule returned validated=False "
          f"(median centred r {DEC_C4.get('median_centred_r', float('nan')):.3f}). A panel built on B3's "
          "rank-matched ESM placeholder would present an unvalidated axis as a prediction.")

## 4. Panel D — the four-group dual-mechanism design (roadmap N3)

The C4 crosstab (destabilising × top-decile rewiring) turned into four **named** lists:
destabilising-only, rewiring-only, both, neither. Without a validated specificity axis only two of
the four groups exist, and those two are exactly Panel S and its controls, so Panel D is not built
a second time.

In [ ]:
PANEL_D = {}
if HAVE_REWIRING:
    groups = {'destabilising_only': A[A.destab & ~A.high_rewiring], 'rewiring_only': A[~A.destab & A.high_rewiring],
              'both': A[A.destab & A.high_rewiring], 'neither': A[~A.destab & ~A.high_rewiring]}
    for name, pool in groups.items():
        key = {'destabilising_only': 'fold_lo', 'both': 'fold_lo', 'rewiring_only': 'rewiring'}.get(name, 'fold_halfwidth')
        asc = key == 'fold_halfwidth'                      # strongest effect first; narrowest interval for 'neither'
        PANEL_D[name] = pool.sort_values(['nonhyper', key], ascending=[False, asc]).head(min(N_PER_PANEL, 6))
    print({k: len(v) for k, v in PANEL_D.items()})
else:
    print('Four-group design not available: only destabilising vs not-destabilising is supported, '
          'which Panel S and its controls already cover.')
PANEL_D_SIZES = {k: len(v) for k, v in PANEL_D.items()}

## 5. Assemble the panel, assign assays and a priority tier

**Tier 1:** seen in a non-hypermutated tumour **and** an interval narrower than the atlas median.
**Tier 2:** everything else that met the panel's rule. A ranking aid for where to start, not a
statement that tier-2 variants matter less.

In [ ]:
MEDIAN_HW = float(A.fold_halfwidth.median())
COLS = ['variant_id', 'gene', 'domain_idx', 'pos', 'pred_ddG_fold', 'fold_lo', 'fold_halfwidth',
        'rewiring', 'nonhyper', 'top_primary_sites', 'mechanism', 'assay_recommendation', 'priority_tier']

def assemble(df, mechanism, assays):
    out = df.copy()
    out['mechanism'] = mechanism; out['assay_recommendation'] = '; '.join(assays)
    out['priority_tier'] = np.where(out.nonhyper & (out.fold_halfwidth <= MEDIAN_HW), 1, 2)
    out = out.reindex(columns=COLS)
    out['domain_idx'] = pd.to_numeric(out.domain_idx, errors='coerce').astype('Int64')   # 2, not 2.000
    return out

parts = [assemble(PANEL_S, 'destabilising (Panel S)', STABILITY_ASSAYS),
         assemble(PANEL_S_CTRL, 'tolerated control (Panel S)', ['same stability assay as Panel S (negative control)']),
         assemble(PANEL_R, 'rewiring only (Panel R)', SPECIFICITY_ASSAYS)]
parts += [assemble(v, f'{k} (Panel D)', STABILITY_ASSAYS + (SPECIFICITY_ASSAYS if k in ('rewiring_only', 'both') else []))
          for k, v in PANEL_D.items()]
CAND = pd.concat([p for p in parts if len(p)], ignore_index=True).drop_duplicates(['variant_id', 'mechanism'])
CAND.to_csv(R.out('experimental_candidates.csv'), index=False)
SUMMARY = CAND.groupby('mechanism').agg(candidates=('variant_id', 'size'),
                                        tier1=('priority_tier', lambda s: int((s == 1).sum()))).reset_index()
SUMMARY.to_csv(R.out('panel_summary.csv'), index=False)
print(SUMMARY.to_string(index=False))

## Results

**What this is not.** A destabilising call here is a hypothesis this stage exists to test, not
evidence on its own. Assay feasibility (expression, peptide availability, cost, throughput) is
not modelled and must be checked before committing bench time. Panel sizes are a first campaign,
to be revised when the first results come back. `experimental_candidates.csv` lists COSMIC
variants with tumour sites, so it stays on Drive (`.gitignore`); the report below omits the
tumour-site column.

In [ ]:
for k, v in [('n_per_panel', N_PER_PANEL), ('specificity_axis_validated', HAVE_REWIRING),
             ('panel_s_destabilising', len(PANEL_S)), ('panel_s_control', len(PANEL_S_CTRL)),
             ('panel_s_control_same_domain', CTRL_MATCHED), ('panel_r_rewiring_only', len(PANEL_R)),
             ('four_group_design_available', HAVE_REWIRING), ('total_candidates', len(CAND)),
             ('total_tier1', int((CAND.priority_tier == 1).sum())), ('median_atlas_fold_halfwidth', MEDIAN_HW)]:
    R.metric(k, v)
for k, v in PANEL_D_SIZES.items(): R.metric(f'panel_d_{k}', v)

R.section('Panel sizes by class', 'Tier 1 = seen in a non-hypermutated tumour and a below-median 90% half-width.', SUMMARY)
R.section('Candidates', 'Full table (with tumour sites): `experimental_candidates.csv`, on Drive only.',
          CAND.drop(columns=['top_primary_sites', 'assay_recommendation']), '{:.3f}')
R.section('Assays by class', '', pd.DataFrame([('stability (Panel S)', '; '.join(STABILITY_ASSAYS)),
                                               ('specificity (Panel R)', '; '.join(SPECIFICITY_ASSAYS))],
                                              columns=['class', 'recommended assays']))
if not HAVE_REWIRING:
    R.section('Why there is no specificity or four-group panel',
              f"C4's pre-registered rule returned validated=False (median centred r "
              f"{DEC_C4.get('median_centred_r', float('nan')):.3f}; wins vs the average profile "
              f"{DEC_C4.get('wins_vs_mean')}/{DEC_C4.get('domains')}). Panel R and the rewiring groups of Panel D are "
              "intentionally empty rather than built from the disclaimed ESM-based label in B3.")
R.finish([f"{len(CAND)} candidates in {SUMMARY.shape[0]} classes, {int((CAND.priority_tier == 1).sum())} of them tier 1.",
          f"Controls from the same domains as the destabilising picks: {CTRL_MATCHED} of {len(PANEL_S_CTRL)}.",
          f"Specificity axis validated (C4): {HAVE_REWIRING}" + ('.' if HAVE_REWIRING else
          ' — no rewiring panel and no four-group design, by design.'),
          'Nothing upstream was re-run or changed; a prioritisation aid, not a result.'])